# 📊 Đánh Giá & Phân Tích Mô Hình
## Mô hình Đánh giá Rủi ro Tín dụng
**Nhóm 14 - Bài tập lớn môn Học Máy**

---

### Nội dung
1. Metrics đánh giá: Accuracy, Precision, Recall, F1, AUC-ROC
2. So sánh mô hình (bảng tổng hợp)
3. ROC Curves & Precision-Recall Curves
4. Confusion Matrix
5. Feature Importance & SHAP (Điểm cộng)
6. Phân tích lỗi (Error Analysis)
7. Learning Curves (Bias-Variance)
8. Kết luận & Đề xuất

## 1. Chuẩn bị

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data.load_data import load_german_credit, load_give_me_credit
from src.data.preprocess import preprocess_pipeline
from src.features.feature_engineering import create_credit_features_german, create_credit_features_gmsc
from src.models.train_model import get_default_models, train_all_models, create_stacking_model
from src.evaluation.evaluate_model import (
    evaluate_model, compare_models, error_analysis,
    compute_learning_curves, generate_report
)
from src.visualization.visualize import (
    plot_roc_curves, plot_precision_recall_curves,
    plot_confusion_matrices, plot_learning_curves,
    plot_feature_importance, plot_shap_analysis, plot_calibration_curves
)

%matplotlib inline
print('✅ Import thành công!')

## 2. Nạp dữ liệu & Huấn luyện lại

In [ ]:
# German Credit
df_german = create_credit_features_german(load_german_credit())
prep_g = preprocess_pipeline(df_german, 'target')
X_train_g, X_test_g = prep_g['X_train'], prep_g['X_test']
y_train_g, y_test_g = prep_g['y_train'], prep_g['y_test']

# Huấn luyện
trained_g = train_all_models(X_train_g, y_train_g)
stacking_g = create_stacking_model()
stacking_g.fit(X_train_g, y_train_g)
trained_g['Stacking Ensemble'] = {'model': stacking_g}

## 3. Đánh giá từng mô hình

In [ ]:
# Đánh giá tất cả mô hình trên German Credit
metrics_g = {}
for name, result in trained_g.items():
    model = result.get('model')
    if model:
        metrics_g[name] = evaluate_model(model, X_test_g, y_test_g, name)

## 4. So sánh mô hình

In [ ]:
# Bảng so sánh
comparison_g = compare_models(trained_g, X_test_g, y_test_g)
display(comparison_g)

## 5. Đường cong ROC

In [ ]:
plot_roc_curves(metrics_g, y_test_g, 'ROC Curves - German Credit', save=False)

## 6. Đường cong Precision-Recall

In [ ]:
plot_precision_recall_curves(metrics_g, y_test_g, 'PR Curves - German Credit', save=False)

## 7. Confusion Matrix

In [ ]:
plot_confusion_matrices(metrics_g, save=False)

## 8. Feature Importance & SHAP (Điểm cộng)

In [ ]:
# Feature Importance từ Random Forest
from src.features.feature_engineering import get_feature_importance

importance_df = get_feature_importance(X_train_g, y_train_g, method='random_forest')
plot_feature_importance(importance_df, top_n=15, save=False)

In [ ]:
# SHAP Analysis (Điểm cộng - kiến thức ngoài chương trình)
# SHAP sử dụng lý thuyết trò chơi Shapley để giải thích mô hình
best_model_name = comparison_g.iloc[0]['Model']
best_model = trained_g[best_model_name]['model']
print(f'Phân tích SHAP cho mô hình tốt nhất: {best_model_name}')

try:
    plot_shap_analysis(best_model, X_test_g, model_name=best_model_name, save=False)
except Exception as e:
    print(f'Lưu ý: {e}')
    print('Cài đặt: pip install shap')

## 9. Phân tích lỗi (Error Analysis)

In [ ]:
# Phân tích lỗi cho mô hình tốt nhất
error_result = error_analysis(
    best_model, X_test_g, y_test_g,
    feature_names=X_train_g.columns.tolist(),
    model_name=best_model_name
)

## 10. Learning Curves (Bias-Variance)

In [ ]:
# Learning curve cho mô hình tốt nhất
lc_data = compute_learning_curves(best_model, X_train_g, y_train_g, cv=5, scoring='f1')
plot_learning_curves(lc_data, model_name=best_model_name, save=False)

## 11. Calibration Curves

In [ ]:
plot_calibration_curves(metrics_g, y_test_g, save=False)

## 12. Báo cáo tổng hợp

In [ ]:
report = generate_report(metrics_g, 'German Credit')

## 13. Kết luận & Đề xuất

### Kết luận:
1. **Mô hình Ensemble** (Random Forest, XGBoost, Stacking) cho kết quả tốt nhất
2. **Logistic Regression** là baseline tốt với khả năng giải thích cao
3. **SMOTE** cải thiện đáng kể Recall cho lớp thiểu số
4. **SHAP** giúp giải thích tại sao mô hình đưa ra quyết định

### Phân tích lỗi:
- **False Negative** (bỏ lọt rủi ro) nguy hiểm hơn False Positive trong tín dụng
- Cần cân nhắc chi phí lỗi khi chọn ngưỡng phân loại

### Đề xuất cải tiến:
1. Thu thập thêm dữ liệu cho lớp thiểu số
2. Thử nghiệm Cost-Sensitive Learning
3. Kết hợp nhiều nguồn dữ liệu (feature fusion)
4. Triển khai monitoring mô hình trong sản xuất